# Day 2 · 01 · Power BI Semantic Model on Databricks

![Power BI report mockup](../../../assets/images/powerbi_report_mockup.png)

How this visual helps: it reminds participants what the Databricks Gold model is ultimately serving: Power BI pages with slicers, KPIs and visuals. Use it as the Day 2 anchor before discussing import mode, DirectQuery and semantic-model contracts.

This is the official Power BI semantic-model module. Day 1 built the Gold model; this notebook decides how Power BI should consume it.

## Program Map for Power BI Developers

Session 2 starts where Power BI developers usually feel the business impact: model shape, refresh behavior, query cost and ownership.

| Module | Status | Goal for Power BI developers | End artifact |
|---|---|---|---|
| Power BI semantic model | Required | choose source tables, relationships, measures and refresh mode | Power BI handoff packet |
| Workshop 2 dataset performance | Required | build serving objects that reduce report cost and support incremental refresh | monthly aggregate, incremental view, KPI table |
| Performance and automation orientation | Required | troubleshoot slow Databricks-backed reports and understand production refresh | operations and certification checklist |
| AI/BI Dashboards and Genie | Optional | compare Databricks-native BI with Power BI for governed exploration | AI/BI checklist |

Expected observation: Day 2 is about consumption design. The same Gold model can produce a fast, trusted semantic model or a slow, expensive one depending on these decisions.

## Business Scenario

RetailHub wants to publish an executive Sales dashboard and a drill-through detail page. The Gold model is ready, but the team must still decide source tables, relationships, measures, refresh mode and handoff responsibilities.

## Power BI Developer Lens

The central question is not “can Power BI connect to Databricks?” It can. The professional question is: **which Databricks object should Power BI consume, under which refresh mode, with which KPI ownership and cost guardrails?**

Keep these decisions explicit:

- model shape: star schema, wide table or serving aggregate,
- measure ownership: Databricks Gold field, Databricks KPI table or DAX measure,
- refresh mode: Import, DirectQuery or Composite,
- troubleshooting path: Performance Analyzer -> SQL Warehouse Query History -> Query Profile,
- ownership: who changes the source object, KPI definition and refresh schedule.

## Learning Objectives

By the end of this notebook you can:

- define a Power BI semantic model and its Databricks dependency,
- choose between star schema and wide table consumption,
- validate relationship readiness and freshness,
- decide which logic belongs in Databricks SQL vs DAX or Power Query,
- explain Import, DirectQuery and Composite modes,
- simulate incremental refresh with a half-open interval,
- prepare a practical Power BI handoff packet.

## Setup

Resolve the participant catalog. `00_setup` executes `USE CATALOG`, so SQL cells can use two-part names such as `gold.fact_sales_dashboard`.

In [0]:
%run ../../../setup/00_setup

### Configuration

Confirm the active catalog and schemas before running the Day 2 examples.

In [0]:
display(spark.createDataFrame([
    ("CATALOG", CATALOG),
    ("BRONZE", BRONZE),
    ("SILVER", SILVER),
    ("GOLD", GOLD),
], ["Variable", "Value"]))

### Runtime Pre-check

Day 2 starts from the Gold model created in Workshop 1. This check fails early if the model is not available.

In [0]:
required_tables = [
    f"{GOLD}.dim_date",
    f"{GOLD}.dim_customer",
    f"{GOLD}.dim_product",
    f"{GOLD}.fact_sales",
    f"{GOLD}.fact_sales_dashboard",
]
missing = [t for t in required_tables if not spark.catalog.tableExists(t)]
if missing:
    for table in missing:
        print("[MISSING]", table)
    raise Exception("Pre-check failed. Run Workshop 1 solution before starting Day 2.")
print("[OK] Day 1 Gold model is available.")

## 1. Gold Inventory

Definition: a Power BI handoff starts with a concrete object inventory. The BI owner needs object names, row counts and the intended usage of each table.

Expected observation: `fact_sales` and `fact_sales_dashboard` should have the same row count because both are line-grain tables.

In [0]:
%sql
SELECT 'gold.dim_date' AS object_name, 'date dimension' AS purpose, COUNT(*) AS rows FROM gold.dim_date
UNION ALL
SELECT 'gold.dim_customer', 'customer dimension', COUNT(*) FROM gold.dim_customer
UNION ALL
SELECT 'gold.dim_product', 'product dimension', COUNT(*) FROM gold.dim_product
UNION ALL
SELECT 'gold.fact_sales', 'star-schema fact at line grain', COUNT(*) FROM gold.fact_sales
UNION ALL
SELECT 'gold.fact_sales_dashboard', 'wide line-grain dashboard table', COUNT(*) FROM gold.fact_sales_dashboard
ORDER BY object_name

## 2. Semantic Model Definition

A **Power BI semantic model** is the governed layer that contains tables, relationships, measures, formatting and refresh behavior. Reports query the semantic model; the semantic model queries Databricks.

Databricks Gold should provide trusted rows and reusable fields. Power BI should provide report-specific relationships, measures and user interactions.

Expected observation: semantic modeling is a contract decision, not only a UI step in Power BI Desktop.

## 3. Star Schema vs Wide Table Decision

![Star schema vs wide table](../../../assets/images/star_schema_vs_flat_table.png)

How this visual helps: it makes the modeling trade-off visible before Power BI is opened. The left side shows the governed star schema used for reusable semantic models; the right side shows a flat/wide table pattern that can work for quick prototypes but repeats dimension attributes and is harder to govern at scale.

A **star schema** separates facts and dimensions. It is the governed default for reusable semantic models because dimensions can filter many facts consistently.

A **wide table** repeats dimension attributes on each fact row. It is useful for prototypes or narrow dashboard pages, but it can scan more data in DirectQuery and can duplicate business logic.

Professional rule: use `gold.fact_sales` plus dimensions for governed semantic models; use `gold.fact_sales_dashboard` for quick report prototypes or serving tables.


In [0]:
%sql
WITH star AS (
  SELECT ROUND(SUM(CASE WHEN f.is_completed THEN f.line_revenue ELSE 0 END), 2) AS revenue
  FROM gold.fact_sales f
  JOIN gold.dim_customer c ON f.customer_id = c.customer_id
  JOIN gold.dim_product p ON f.product_id = p.product_id
),
wide AS (
  SELECT ROUND(SUM(CASE WHEN is_completed THEN line_revenue ELSE 0 END), 2) AS revenue
  FROM gold.fact_sales_dashboard
)
SELECT
  star.revenue AS star_schema_revenue,
  wide.revenue AS wide_table_revenue,
  ROUND(star.revenue - wide.revenue, 2) AS revenue_delta
FROM star CROSS JOIN wide

Expected observation: the revenue delta should be zero. The star schema and wide table are different consumption shapes, not different business definitions.

## 4. Relationship Readiness

**Relationship readiness** means dimension keys are unique and fact rows can match those keys. Power BI many-to-one relationships depend on this condition.

An **orphan check** finds fact rows without matching dimensions. Orphans usually become blanks in slicers and can confuse users.

In [0]:
%sql
SELECT 'dim_date.date_key uniqueness' AS check_name, COUNT(*) - COUNT(DISTINCT date_key) AS issue_rows FROM gold.dim_date
UNION ALL
SELECT 'dim_customer.customer_id uniqueness', COUNT(*) - COUNT(DISTINCT customer_id) FROM gold.dim_customer
UNION ALL
SELECT 'dim_product.product_id uniqueness', COUNT(*) - COUNT(DISTINCT product_id) FROM gold.dim_product
UNION ALL
SELECT 'fact_sales -> dim_date orphans', COUNT(*) FROM gold.fact_sales f LEFT ANTI JOIN gold.dim_date d ON f.order_date = d.date_key
UNION ALL
SELECT 'fact_sales -> dim_customer orphans', COUNT(*) FROM gold.fact_sales f LEFT ANTI JOIN gold.dim_customer c ON f.customer_id = c.customer_id
UNION ALL
SELECT 'fact_sales -> dim_product orphans', COUNT(*) FROM gold.fact_sales f LEFT ANTI JOIN gold.dim_product p ON f.product_id = p.product_id

Expected observation: every issue count should be zero before building Power BI relationships.

## 5. Databricks SQL vs DAX vs Power Query

| Logic | Best home | Reason |
|---|---|---|
| joins to conformed dimensions | Databricks Gold | shared, expensive and reusable |
| unknown labels and source quality flags | Databricks Gold | consistent across BI tools |
| additive base fields | Databricks Gold fields | easy to validate and reuse |
| report-specific measures | Power BI DAX | interactive and presentation-aware |
| time intelligence formatting | Power BI DAX | depends on report calendar behavior |
| large row-by-row transformations | Databricks SQL | avoid Power Query work over DirectQuery |
| cosmetic report-only renames | Power BI | does not change shared data contract |

Expected observation: the decision is about ownership. Shared logic belongs closer to Gold; visual logic belongs in the semantic model.

## 6. KPI Dictionary and Starter Measures

A **KPI dictionary** defines each measure in business language before report authors build visuals. Ambiguous filters are the fastest way to create inconsistent dashboards.

A **DAX measure** is a reusable calculation evaluated at query time in the current filter context. Measures should express business logic such as revenue, margin rate or return rate.

In [0]:
%sql
SELECT 'Revenue' AS kpi_name, 'SUM line_revenue where is_completed = true' AS definition, 'Databricks field + DAX measure' AS owner
UNION ALL
SELECT 'Gross Margin', 'SUM line_margin where is_completed = true', 'Databricks field + DAX measure'
UNION ALL
SELECT 'Completed Orders', 'DISTINCTCOUNT order_id where is_completed = true', 'DAX measure'
UNION ALL
SELECT 'Margin Rate %', 'Gross Margin divided by Revenue with divide-by-zero protection', 'DAX measure'
UNION ALL
SELECT 'Return Rate %', 'Returned orders divided by completed plus returned orders', 'DAX measure or certified KPI table' 

Starter DAX measures:

```DAX
Revenue =
CALCULATE (
    SUM ( fact_sales[line_revenue] ),
    fact_sales[is_completed] = TRUE ()
)

Gross Margin =
CALCULATE (
    SUM ( fact_sales[line_margin] ),
    fact_sales[is_completed] = TRUE ()
)

Completed Orders =
CALCULATE (
    DISTINCTCOUNT ( fact_sales[order_id] ),
    fact_sales[is_completed] = TRUE ()
)

Margin Rate % =
DIVIDE ( [Gross Margin], [Revenue] )
```

Expected observation: Databricks supplies governed fields and filters; DAX defines reusable report measures.

## 7. Import, DirectQuery and Composite

![Import vs DirectQuery](../../../assets/images/import_vs_directquery.png)

`Import` copies data into Power BI memory during refresh. It is usually fastest for dashboards and safest for cost.

`DirectQuery` sends each interaction back to the SQL Warehouse. It is useful for live requirements, but it can multiply queries through visuals and slicers.

`Composite` mixes Import and DirectQuery tables. It solves mixed freshness requirements, but it increases model complexity.

In [0]:
%sql
SELECT 'Executive dashboard' AS report_area, 'Import' AS recommended_mode, 'fast interactions and scheduled refresh' AS reason
UNION ALL
SELECT 'Operational live monitor', 'DirectQuery', 'freshness requirement is stronger than latency risk'
UNION ALL
SELECT 'Hybrid management report', 'Composite', 'import dimensions and aggregates, query live detail only when needed'
UNION ALL
SELECT 'Ad-hoc analyst exploration', 'Import or Databricks SQL', 'avoid uncontrolled DirectQuery fan-out from many visuals' 

## 8. Incremental Refresh Definition

![Incremental refresh range](../../../assets/images/incremental_refresh_range.png)

How this visual helps: it makes the `RangeStart` / `RangeEnd` rule concrete. Participants can see that old partitions are kept, only the refresh window is reloaded, and `RangeEnd` must stay exclusive to avoid overlapping rows.

**Incremental refresh** is a Power BI Service refresh policy for **Import** semantic models. It does not mean that Power BI magically detects every new row by itself, and it does not make Databricks tables incremental. It means Power BI stores the imported table in date/time partitions and, after publishing, refreshes only the partitions required by the policy.

What happens in practice:

1. In Power BI Desktop you create `RangeStart` and `RangeEnd` parameters.
2. You filter the fact table on a stable Date/Time column, for example `order_datetime >= RangeStart AND order_datetime < RangeEnd`.
3. You publish the model to Power BI Service and define the policy, for example: store 2 years, refresh last 30 days.
4. The first Service refresh loads the historical store period into partitions.
5. Later scheduled refreshes normally refresh only the configured recent window, not the whole historical table.

So the precise answer is: **Power BI refreshes its imported partitions incrementally**. Databricks still receives SQL queries for the requested date windows. If query folding works, those date filters are pushed to the SQL Warehouse, so less data is scanned/transferred and fewer rows enter the Power BI model during each refresh.

| Question | Practical answer |
|---|---|
| Does it incrementally refresh Power BI? | Yes — in Power BI Service, for imported semantic-model partitions. |
| Does it update Databricks incrementally? | No — Databricks is the source. Its own incremental pipelines are separate. |
| Does it load only a selected period? | Yes — each refresh partition is generated from a date/time window. |
| Why does folding matter? | Without folding, Power Query may pull too much data first and filter later, which defeats the performance benefit. |
| Why use a half-open interval? | `>= RangeStart` and `< RangeEnd` prevents duplicate rows at partition boundaries. |

For this course, `gold.v_fact_sales_incremental` exists to expose a refresh-friendly Date/Time column and stable Gold logic. Power BI should filter that column with `RangeStart` and `RangeEnd`; Databricks should see a SQL query with the same date predicate in Query History.

Documentation: [Configure incremental refresh](https://learn.microsoft.com/en-us/power-bi/connect-data/incremental-refresh-configure), [Incremental refresh overview](https://learn.microsoft.com/en-us/power-bi/connect-data/incremental-refresh-overview).

Expected observation: half-open intervals avoid overlapping rows between adjacent refresh windows, and folding turns the Power BI refresh policy into filtered SQL on the SQL Warehouse.


In [0]:
%sql
WITH refresh_window AS (
  SELECT DATE '2024-01-01' AS range_start, DATE '2024-04-01' AS range_end
)
SELECT
  range_start,
  range_end,
  COUNT(*) AS line_rows,
  COUNT(DISTINCT order_id) AS orders,
  ROUND(SUM(CASE WHEN is_completed THEN line_revenue ELSE 0 END), 2) AS revenue
FROM gold.fact_sales_dashboard CROSS JOIN refresh_window
WHERE order_date >= range_start
  AND order_date < range_end
GROUP BY range_start, range_end

## 9. Query Folding Definition

**Query folding** means Power Query translates supported transformation steps into a native query that the source system can execute. With Databricks, the goal is simple: filters, selected columns, joins and aggregations should be pushed to the SQL Warehouse instead of being performed locally by the Power Query engine after too much data has already been retrieved.

Why it matters for Power BI on Databricks:

| Scenario | If folding works | If folding breaks |
|---|---|---|
| Import refresh | SQL Warehouse receives a filtered/selective query | Power Query may retrieve more data and transform locally |
| Incremental refresh | `RangeStart` / `RangeEnd` become source-side date predicates | Power BI may not get the expected partition-refresh benefit |
| DirectQuery | visuals generate SQL that Databricks can execute efficiently | report latency rises or transformations become unsupported |
| Cost control | fewer rows/columns are scanned and transferred | SQL Warehouse and refresh time increase unnecessarily |

How to use it in real Power BI work:

1. Start from a curated Databricks Gold table or view, not raw Silver tables.
2. Apply row filters and column selection early in Power Query.
3. For incremental refresh, filter a Date/Time column with `RangeStart` and `RangeEnd` using a half-open interval.
4. Keep Power Query steps boring: filter rows, choose columns, rename columns, change simple types.
5. Push complex shaping into Databricks SQL views or Gold/serving tables: joins, heavy calculated columns, parsing, deduplication, business rules.
6. In Power Query Desktop, right-click an applied step and check **View Native Query** when available.
7. After publishing, confirm from Databricks SQL Warehouse Query History that the executed query contains the expected filters and selected columns.

Practical rule: Power Query is allowed to prepare the model; Databricks should do the heavy data work. If a transformation is business-critical, expensive or shared by multiple reports, move it into Gold or a serving view.

Documentation: [Query folding basics](https://learn.microsoft.com/en-us/power-query/query-folding-basics), [Query folding indicators](https://learn.microsoft.com/en-us/power-query/step-folding-indicators), [Troubleshoot incremental refresh](https://learn.microsoft.com/en-us/power-bi/connect-data/incremental-refresh-troubleshoot).

Expected observation: filters and aggregation should appear in the Databricks SQL plan. If a Power Query step cannot fold, redesign it in Gold or in a serving view.


In [0]:
%sql
EXPLAIN FORMATTED
SELECT
  category,
  ROUND(SUM(CASE WHEN is_completed THEN line_revenue ELSE 0 END), 2) AS revenue
FROM gold.fact_sales_dashboard
WHERE order_date >= DATE '2024-01-01'
  AND order_date < DATE '2025-01-01'
GROUP BY category
ORDER BY revenue DESC

Expected observation: filters and aggregation should appear in the Databricks SQL plan. If a Power Query step cannot fold, redesign it in Gold or in a serving view.

## 10. Freshness and Row Count Checks

Freshness tells report owners how current the data is. Row-count trend checks catch unexpected drops before a scheduled refresh publishes an empty or partial dataset.

In [0]:
%sql
SELECT
  MIN(order_date) AS min_order_date,
  MAX(order_date) AS max_order_date,
  COUNT(*) AS line_rows,
  COUNT(DISTINCT order_id) AS orders
FROM gold.fact_sales_dashboard

## 11. Connector Walkthrough

![Power BI connection walkthrough](../../../assets/images/powerbi_connection_walkthrough.png)

Power BI needs the Databricks SQL Warehouse hostname and HTTP path. Authentication depends on workspace policy: PAT, OAuth or Microsoft Entra ID.

Trainer note: open SQL Warehouses -> Connection details during delivery and show where those two values come from.

## 12. Power BI on Databricks Anti-patterns and Approval Checklists

These are the problems that most often create slow, expensive or inconsistent Power BI reports on Databricks.

| Anti-pattern | Why it hurts | Better pattern |
|---|---|---|
| DirectQuery on a wide line-grain table with many slicers | every interaction fans out into expensive SQL | Import aggregates; reserve DirectQuery for justified live detail |
| importing Silver instead of Gold | report authors recreate business rules independently | use certified Gold or serving objects |
| no date table | time intelligence and incremental refresh become fragile | use `gold.dim_date` and a stable date/timestamp column |
| different Revenue definitions across reports | executives see conflicting KPI totals | publish KPI dictionary and certified measures |
| heavy Power Query transformations over DirectQuery | folding may fail and latency increases | push shaping into Databricks Gold or serving views |
| counting orders as rows | line-grain facts overcount orders | use `COUNT(DISTINCT order_id)` with explicit filters |
| aggregate not reconciled to detail | summary and drill-through disagree | reconcile aggregate totals to the certified detail source |

Dataset readiness checklist:

- grain is documented,
- owner is known,
- row counts and freshness are validated,
- relationships are many-to-one ready,
- KPI rules are approved,
- PII and restricted fields are handled,
- refresh expectation is documented.

DirectQuery approval checklist:

- live freshness is a real business requirement,
- source object is narrow or aggregated enough,
- expected visual count and slicer behavior are controlled,
- SQL Warehouse cost guardrails are agreed,
- Query History will be reviewed after publishing,
- Import or Composite was considered first.

Expected observation: DirectQuery is a design exception, not the default answer to freshness concerns.

## 13. Questions to Ask the Databricks/Data Team

Before building or certifying a Power BI semantic model, ask these questions and capture the answers in the handoff packet.

| Question | Why Power BI cares |
|---|---|
| What is the grain of each source object? | prevents duplicated facts and wrong relationships |
| Who owns the Gold or serving object? | identifies who approves schema and KPI changes |
| What is the freshness SLA? | drives Import refresh schedule and DirectQuery justification |
| Is DirectQuery allowed for this object? | controls SQL Warehouse cost and user experience |
| Which KPI definitions are certified? | prevents conflicting DAX measures across reports |
| Does the data contain PII or restricted fields? | affects row/column security and sharing |
| What is the refresh failure escalation path? | prevents silent stale reports |

Expected observation: this is the practical bridge between a BI developer and the Databricks platform team.

## 14. Power BI Handoff Packet

A professional handoff is a packet of decisions, not just a connection string.

| Handoff item | Required decision |
|---|---|
| Connection details | SQL Warehouse hostname and HTTP path from Databricks UI |
| Authentication | PAT, OAuth or Entra ID pattern chosen by workspace policy |
| Source option | star schema for governed model, dashboard table for quick prototype |
| Relationship map | fact date/customer/product keys to unique dimensions |
| Model mode | Import by default, DirectQuery only with explicit freshness need |
| Refresh expectation | schedule, owner and freshness SLA |
| Cost guardrails | warehouse size, auto-stop, visual count and DirectQuery limits |
| Measure ownership | which measures are DAX-owned vs Gold-owned |

Expected observation: this table is the conversation checklist between Databricks and Power BI owners.

## 15. Performance Analyzer Awareness

Power BI Performance Analyzer shows which visuals trigger slow DAX queries or source queries. For Databricks DirectQuery, pair it with SQL Warehouse Query History.

Expected observation: a slow report page is investigated from both sides: Power BI visual fan-out and Databricks SQL execution profile.

---
## 16. Live Demo: Power BI Desktop Connection (15 min)

**Goal:** show live how to connect Power BI Desktop to a SQL Warehouse and build the first report from the Gold layer.

**Trainer note:** run the steps below live on the projector. Participants observe and take notes/screenshots.


### Step 1: Get Connection Details

**UI path:** SQL Warehouses > [warehouse name] > Connection details

Required values:
* **Server hostname:** for example `adb-1234567890.1.azuredatabricks.net`
* **HTTP path:** for example `/sql/1.0/warehouses/abc123def456`

Trainer note: show this live in the workspace. Participants should take screenshots.

### Step 2: Power BI Desktop - Get Data

**Power BI path:** Get Data > Azure > Azure Databricks

| Field | Value | Notes |
|------|-------|-------|
| Server Hostname | copied from Connection details | without `https://` |
| HTTP Path | copied from Connection details | full path |
| Data Connectivity mode | **Import** by default | show the difference below |


### Import vs DirectQuery - Decision Table

| Criterion | Import | DirectQuery |
|-----------|--------|-------------|
| Data refresh | Scheduled refresh, for example 4x/day | Live query on every interaction |
| Report performance | Fast because data is in Power BI memory | Slower because each interaction queries the warehouse |
| SQL Warehouse cost | Mostly during refresh | During report usage and interactions |
| Data freshness | Delayed until the next refresh | Near real-time |
| Dataset size | Import model limits apply | No imported dataset size limit |
| Typical use | Executive dashboards, daily reports | Live monitoring, very large datasets |

**Recommendation for most cases:** Import with scheduled refresh every 4-8 hours.

**Use DirectQuery only when:** data must be live or the dataset cannot reasonably fit the Import model.


### Step 3: Select Tables and Build the Model

**Demo flow:**
1. Navigator > select catalog `retailhub_trainer` > schema `gold`
2. Select: `dim_customer`, `dim_product`, `dim_date`, `fact_sales`
3. Load using Import mode

### Step 4: Relationships

Power BI should detect relationships automatically because dimension keys are unique:
* `fact_sales.customer_id` -> `dim_customer.customer_id` (Many:1)
* `fact_sales.product_id` -> `dim_product.product_id` (Many:1)
* `fact_sales.order_date` -> `dim_date.date_key` (Many:1)

**Warning:** if Power BI does not detect relationships automatically, treat it as a grain or orphan-key problem. Show that the Unknown member pattern from Workshop 1 resolves this issue.

### Step 5: First Visual (2 min)

1. Stacked bar: Revenue by Category
2. Slicer: Region
3. Card: Total Revenue

**Point to emphasize:** this works because the Gold layer has correct grain and zero orphan keys.

### Step 6: Query Folding Check in Power BI Desktop

**Why:** before enabling Incremental Refresh, confirm that the date filter can be pushed down to Databricks as SQL. Otherwise Power BI may retrieve too much data and filter locally.

**Demo flow:**
1. Transform data > select the fact/view query
2. Keep steps simple: selected columns, types, date filter
3. Right-click the filter step > **View Native Query**
4. Show that the date predicate should be visible in the SQL
5. After publishing, confirm in Databricks Query History that the SQL Warehouse received a query with the date predicate

**Rule:** if `View Native Query` disappears after a step, treat that step as suspicious. Move heavy transformations to Databricks Gold or a serving view.

### Step 7: Incremental Refresh (optional demo)

In Power BI Service you can configure Incremental Refresh for Import tables:
1. Create `RangeStart` and `RangeEnd` parameters (DateTime) in Power Query Editor
2. Filter the fact/view by `order_datetime >= RangeStart AND order_datetime < RangeEnd`
3. Check folding for the filter step
4. Right-click the table > Incremental refresh > for example Archive 2 years, Incremental 30 days
5. Publish to Power BI Service
6. The first refresh loads the historical period; later refreshes refresh only the partitions defined by the incremental policy

**What this really means:**
* Power BI Service refreshes model partitions, not the whole table from scratch
* Databricks SQL Warehouse receives queries for specific date windows
* If folding works, the date filter runs in Databricks, so less data is scanned and transferred to Power BI
* If folding does not work, incremental refresh loses its performance value because Power Query may filter too late

**Why `v_fact_sales_incremental` from Workshop 2 matters:**
* exposes a stable DateTime column for `RangeStart` / `RangeEnd`
* hides complex Gold logic on the Databricks side
* supports query folding because Power BI applies a simple time-column filter
* lets you confirm in Query History that refresh queries only the required period


---
## 17. Live Demo: Delta Sharing (15 min)

### What is Delta Sharing?

An open protocol for securely sharing data outside the organization without copying it.

| Feature | Delta Sharing | Traditional approach |
|-------|--------------|---------------------|
| Data copying | No - recipient reads from your storage | Yes - CSV/Parquet export |
| Governance | Full control: what, to whom, when | No control after file delivery |
| Freshness | Always reads the latest shared version | Fixed at export time |
| Format | Open protocol, Parquet over REST | Varies by export |
| Recipients | Power BI, pandas, Spark, Tableau, any compatible client | Only tools that can read the exported format |

### Use Cases

* Share Gold KPIs with an external business partner
* Headquarters shares governed data with business units or other workspaces
* A data science team consumes curated BI data
* Compliance: an auditor gets read-only access without file copies


In [0]:
%sql
-- ============================================================
-- DELTA SHARING: Live Demo (trainer only)
-- ============================================================
-- Warning: requires CREATE SHARE privileges at the metastore level.
-- Participants observe this demo; they do not run it independently.

-- Step 1: Create the share
CREATE SHARE IF NOT EXISTS retailhub_partner_share
COMMENT 'Gold KPI tables shared with external Power BI consumers';

-- Step 2: Add tables to the share
ALTER SHARE retailhub_partner_share ADD TABLE gold.fact_sales_dashboard_monthly;
ALTER SHARE retailhub_partner_share ADD TABLE gold.dim_customer;
ALTER SHARE retailhub_partner_share ADD TABLE gold.dim_product;

-- Step 3: Inspect share contents
SHOW ALL IN SHARE retailhub_partner_share;


In [0]:
%sql
-- Step 4: Create a recipient for the external consumer
CREATE RECIPIENT IF NOT EXISTS partner_analytics_team
COMMENT 'External analytics team consuming Gold KPIs via Power BI';

-- Step 5: Grant access
GRANT SELECT ON SHARE retailhub_partner_share
TO RECIPIENT partner_analytics_team;

-- Step 6: Get the activation link
-- UI path: Sharing > Recipients > partner_analytics_team > Activation link
-- The recipient uses this link to connect from Power BI or another compatible tool.

SHOW RECIPIENTS;


### How Does the Recipient Consume a Delta Share in Power BI?

| Option | When to use | Requirements |
|-------|-------------|----------|
| **A - Databricks connector** | Recipient has access to the workspace | Databricks account, Get Data > Azure Databricks |
| **B - Delta Sharing connector** | External recipient without Databricks workspace access | Power BI Desktop + `.share` credentials file |
| **C - Fabric Dataflow Gen2** | Organization uses Fabric | Fabric capacity, endpoint URL + bearer token |

**Option B in detail - most common for partners:**
1. Recipient gets the activation link from the Databricks UI
2. Recipient saves the credentials file (`.share`)
3. In Power BI: Get Data > Delta Sharing
4. Select the `.share` file
5. Choose tables from the list of shared objects

### Key Takeaways

* Delta Sharing means sharing without copying; governance stays with the provider
* The recipient does not need a Databricks workspace because the protocol is open
* Good fit for partners, auditors and teams in another workspace
* Connects naturally to Power BI through a dedicated connector


## Summary and Workshop Handoff

You now have the semantic model vocabulary and the contract for Workshop 2: build smaller, refresh-ready, performance-aware objects for Power BI without changing the Day 1 Gold model.